In [ ]:
import os
import json
import pandas as pd
import numpy as np
import pylab as plt
import seaborn as sns
from collections import Counter
from adjustText import adjust_text

In [ ]:
%load_ext autoreload
%autoreload 2
import src.count_utils as utils

In [ ]:
import jupyter_black
jupyter_black.load()

In [ ]:
plt.style.use("../src/mpl_style.txt")

### Subgroup Analysis on Journals and Disciplines

In [ ]:
sections = ["abstract", "introduction", "methods", "results", "discussion", "full"]
colors = dict(zip(sections, sns.color_palette("colorblind", len(sections))))

In [ ]:
VERSION = "research-article_aimrd_f"
BASELINE = "baseline_2026-01-23"
RESULTS_PATH = os.path.join("../data/results/", BASELINE, VERSION)
secs = next(os.walk(RESULTS_PATH))[1]

In [ ]:
with open(os.path.join(RESULTS_PATH, "filters.json")) as f:
    filters = json.load(f)
journals = filters["all_journals"]

In [ ]:
journals

In [ ]:
journal_names = list(set(journals))
len(journal_names)

In [ ]:
journal_dict = {}
for j in journal_names:
    journal_dict[j] = utils.find_discipline(j)

In [ ]:
journal_dict

In [ ]:
disciplines = [journal_dict[j] for j in journals]

In [ ]:
c = Counter(disciplines)
c

In [ ]:
len([l for l in disciplines if l == None]) / len(journals)

In [ ]:
[l for l, n in c.items() if n > 5000 and l != None]

In [ ]:
freqs_dfs = {}

for sec in secs:
    freqs_dfs[sec] = utils.get_discipline_frequency(RESULTS_PATH, sec)
    p = freqs_dfs[sec]["projection"]
    q = freqs_dfs[sec]["frequency"]
    totals = freqs_dfs[sec]["paper_counts"]
    reg_se = freqs_dfs[sec]["regression se"]
    freqs_dfs[sec]["regression se (ds)"] = list(
        map(utils.se, p, q, totals, ["regression"] * len(p), reg_se)
    )

problematic disciplines (full section):
- computation yields q = 1 in multiple years, 2018 and 2025
- environment has a lower outlier in 2018, making the prediction extremely steep st each timepoint after 2022 has projection over 1
- nutrition: projection > frequency for 2023, 0.58 SE !!!
- psychology usage 2025 < 2024

-> repeat with subsuboptimal thresholds

In [ ]:
freqs_dfs["full"][freqs_dfs["full"]["cutoff"] == "computation"]

In [ ]:
freqs_dfs["full"][freqs_dfs["full"]["cutoff"] == "cancer"]

In [ ]:
freqs_dfs["full"][freqs_dfs["full"]["cutoff"] == "computation"]

In [ ]:
df = pd.concat(freqs_dfs, names=["section"]).reset_index(level=0).reset_index(drop=True)
df = df.rename({"cutoff": "discipline"}, axis=1)
df["section"] = pd.Categorical(
    df["section"],
    categories=[
        "abstract",
        "introduction",
        "methods",
        "results",
        "discussion",
        "full",
    ],
    ordered=True,
)
df

In [ ]:
df_filtered = df[
    (df["time"].between(2022, 2025)) & (df["section"].isin(["abstract", "full"]))
]
df_filtered

In [ ]:
disciplines = df_filtered["discipline"].unique()
n = len(disciplines)

ncols = 5
nrows = int(np.ceil(n / ncols))

fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(ncols, nrows),
    sharex=True,
    sharey=True,
    layout="constrained",
)
axes = axes.flatten()

for i, dis in enumerate(disciplines):
    ax = axes[i]
    sub = df_filtered[df_filtered["discipline"] == dis]

    for section in ["abstract", "full"]:
        sec_data = sub[sub["section"] == section].sort_values("time")

        ax.errorbar(
            sec_data["time"],
            sec_data["usage estimate"],
            yerr=sec_data["regression se (ds)"],
            color=colors[section],
            marker="o",
            linestyle="-",
            capsize=2,
            markersize=2.5,
            clip_on=True,
            label=section,
        )

    # Title
    ax.set_title(dis)

    # Max usage annotation
    max_usage = sub["usage estimate"].max()
    ax.text(
        0.75,  # 0.05,
        0.05,  # 0.9,
        f"{max_usage:.2f}",
        transform=ax.transAxes,
        bbox=dict(
            facecolor="#666666", edgecolor="none", alpha=0.5, boxstyle="round,pad=.2"
        ),
    )
    ax.set_xticks(range(2022, 2026))
    ax.set_ylim([0, 1])

    if i % ncols == 0:
        ax.set_ylabel("LLM-use")
    if i >= ncols * (nrows - 1):
        ax.set_xticklabels(range(2022, 2026), rotation=60)

axes[0].legend()

# plt.tight_layout()
plt.savefig(
    os.path.join(
        "../results/plots",
        BASELINE,
        VERSION,
        "disciplines_individual.png",
    ),
    dpi=300,
)
plt.show()

In [ ]:
df_full = df[(df["section"] == "full") & (df["time"].isin([2024, 2025]))]
df_full = df_full.pivot(
    index="discipline", columns="time", values="usage estimate"
).reset_index()
df_full

In [ ]:
fig, ax = plt.subplots(
    figsize=(3, 3),
    layout="constrained",
)

ax.scatter(df_full[2024], df_full[2025])

texts = []
for _, row in df_full.iterrows():
    texts.append(ax.text(row[2024], row[2025], row["discipline"], fontsize=5))

adjust_text(texts, ax=ax)

ax.set_xlabel("LLM-use (2024)")
ax.set_ylabel("LLM-use (2025)")
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])
ax.legend()

plt.savefig(
    os.path.join(
        "../results/plots",
        BASELINE,
        VERSION,
        "disciplines_2024v2025.png",
    ),
    dpi=300,
)
plt.show()

In [ ]:
df_filtered = df[df["section"] == "full"]

disciplines = df_filtered["discipline"].unique()
n = len(disciplines)

ncols = 5
nrows = int(np.ceil(n / ncols))

fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(ncols, nrows),
    sharex=True,
    sharey=True,
    layout="constrained",
)
axes = axes.flatten()

for i, dis in enumerate(disciplines):
    ax = axes[i]
    sub = df_filtered[df_filtered["discipline"] == dis].sort_values("time")

    ax.plot(
        sub["time"],
        sub["frequency"],
        color=colors["full"],
        label="($q$)",
    )

    ax.plot(
        sub["time"],
        sub["projection"],
        # yerr=sub["regression se"],
        color=colors["full"],
        linestyle="--",
        label="($\\hat{p}_{human}$)",
    )

    ax.fill_between(
        sub["time"],
        sub["projection"] - sub["regression se"],
        sub["projection"] + sub["regression se"],
        alpha=0.2,
        color=colors["full"],
    )

    # Title
    ax.set_title(dis)

    ax.set_ylim([0.6, 1])
    ax.set_xticks(range(2019, 2026, 2))

    if i % ncols == 0:
        ax.set_ylabel("Frequency")
    if i >= ncols * (nrows - 1):
        ax.set_xticklabels(range(2019, 2026, 2), rotation=60)


axes[0].legend()

# plt.tight_layout()
plt.savefig(
    os.path.join(
        "../results/plots",
        BASELINE,
        VERSION,
        "disciplines_individual_freqs.png",
    ),
    dpi=300,
)
plt.show()